# RetailRocket — Sessionisation & Feature Engineering

**Goal:** Transform the raw event log into a tabular dataset with **one row per session** and meaningful features for purchase prediction.

## Session definition
A session ends when a visitor is **inactive for more than 30 minutes** (industry-standard Google Analytics rule).

## Contents
1. Load & assign session IDs  
2. Session-level statistics  
3. Build feature table  
4. Class balance & export

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os

sys.path.append(os.path.join('..', 'src'))
from session_utils import load_events, assign_session_ids, build_session_features

sns.set_theme(style='whitegrid', palette='muted')
DATA_DIR = os.path.join('..', 'data')

## 1. Load & Assign Session IDs

In [ ]:
df = load_events(os.path.join(DATA_DIR, 'events.csv'))
print(f'Loaded {len(df):,} events')
df.head()

In [ ]:
df = assign_session_ids(df)

n_sessions = df['session_id'].nunique()
n_visitors = df['visitorid'].nunique()
print(f'Sessions  : {n_sessions:,}')
print(f'Visitors  : {n_visitors:,}')
print(f'Sessions / visitor (avg): {n_sessions / n_visitors:.2f}')

## 2. Session-Level Statistics

In [ ]:
events_per_session = df.groupby('session_id').size()

print('Events per session:')
print(events_per_session.describe().round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
events_per_session.clip(upper=30).hist(bins=30, ax=ax, color='#4C72B0', edgecolor='white')
ax.set_xlabel('Events per session (capped at 30)')
ax.set_ylabel('Number of sessions')
ax.set_title('Session length distribution')
plt.tight_layout()
plt.show()

## 3. Build Feature Table

In [ ]:
sessions = build_session_features(df)
print(f'Shape: {sessions.shape}')
sessions.head()

In [ ]:
sessions.describe().round(2)

In [ ]:
# Feature distributions (log scale for skewed data)
features = ['n_events', 'n_views', 'n_addtocart', 'n_items', 'duration_sec']

fig, axes = plt.subplots(1, len(features), figsize=(18, 4))
for ax, col in zip(axes, features):
    data = sessions[col].clip(upper=sessions[col].quantile(0.99))
    ax.hist(data, bins=40, color='#4C72B0', edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel('')

plt.suptitle('Feature distributions (99th-percentile clipped)', y=1.02)
plt.tight_layout()
plt.show()

## 4. Class Balance & Export

In [ ]:
n_total    = len(sessions)
n_purchase = sessions['purchased'].sum()
n_no_purchase = n_total - n_purchase
pct = n_purchase / n_total * 100

print('=== Class Balance ===')
print(f'Total sessions      : {n_total:,}')
print(f'Purchased (class=1) : {n_purchase:,} ({pct:.2f} %)')
print(f'No purchase (class=0): {n_no_purchase:,} ({100-pct:.2f} %)')
print(f'Imbalance ratio     : 1 : {n_no_purchase // n_purchase}')

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['No purchase (0)', 'Purchase (1)'], [n_no_purchase, n_purchase],
       color=['#4C72B0', '#55A868'])
ax.set_title('Class balance (target: purchased)')
ax.set_ylabel('Number of sessions')
for i, v in enumerate([n_no_purchase, n_purchase]):
    ax.text(i, v * 1.01, f'{v:,}\n({v/n_total*100:.1f}%)', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with target
corr = sessions[features + ['has_addtocart', 'purchased']].corr()['purchased'].drop('purchased').sort_values(ascending=False)
print('Correlation with target (purchased):')
print(corr.round(3))

In [ ]:
# Export
out_path = os.path.join(DATA_DIR, 'sessions_features.parquet')
sessions.to_parquet(out_path, index=False)
print(f'Saved to {out_path}')
print(f'Final shape: {sessions.shape}  →  {sessions.shape[1]-1} features + 1 target')

## Summary

| Metric | Value |
|--------|-------|
| Total sessions | 1,761,675 |
| Features | 7 (`n_events`, `n_views`, `n_addtocart`, `n_items`, `duration_sec`, `has_addtocart`, `visitorid`) |
| Target | `purchased` (binary) |
| Purchase rate | **0.81 %** — strongly imbalanced |

**Next step → notebook 03: Classification with imbalanced data handling**